# 02 · NeuroLoc-Δ: embedding ESM-2, modello multi-fonte e multi-task, varianti

**Domande critiche e risposte che guidano la metodologia**

1. *Le fonti di localizzazione sono tutte vere allo stesso modo?* No. Il modello stima la localizzazione **latente**
   e osserva ogni fonte attraverso un canale con sensibilità e specificità per classe, apprese
   (stile Dawid-Skene). DeepLoc, sperimentale e curato, è l'ancora con canale perfetto. Così le etichette deboli
   (UniProt per similarità, HPA Approved, mappe neuronali di topo) aggiungono informazione senza imporre i loro errori.
2. *Un compartimento non annotato è un negativo?* Spesso no (UniProt è incompleto): lo assorbe la sensibilità della fonte.
3. *Una proteina neuronale sta dove sta in HeLa?* A livello di organello quasi sempre sì; ma il neurone ha assone,
   dendriti, pre- e post-sinapsi che le linee cellulari non hanno → **testa neuronale** separata, addestrata solo
   sulle proteine espresse nei neuroni.
4. *Le colonne dell'Excel di FFFFF vanno in input?* No (derivano dalle etichette o non cambiano col mutante).
   Diventano **target ausiliari**: il modello deve prevedere dalla sequenza peptide segnale, eliche TM, ancore
   lipidiche, NLS, glicosilazione, aggregazione. Regolarizzano la rappresentazione e, sul mutante, dicono quale
   meccanismo si rompe. Il contesto genico dell'Excel entra solo nel modello delle varianti.
5. *L'embedding medio sente una singola sostituzione?* Quasi per niente. Per le varianti si aggiunge una
   **vista locale** (±15 residui) e il confronto WT/mutante si fa lì.
6. *La mislocalizzazione patogenica passa dai segnali di sorting?* Poco: secondo Lacoste et al. (Cell 2024) passa
   soprattutto da stabilità e inserzione in membrana. Quindi si misurano anche l'LLR di ESM (proxy di stabilità)
   e il ΔΔG di inserzione in membrana con la scala biologica di Hessa, solo dentro i tratti TM.
7. *ClinVar patogenico = mislocalizzato?* No (circa 1 su 6). ClinVar è supervisione debole; la verità sperimentale
   (`varianti_sperimentali`) è usata solo per valutare.
8. *Come misurare il guadagno senza contaminare il test?* Pool `oro+fonti` = stesse proteine di DeepLoc, più
   fonti di etichette. Il confronto con lo stato dell'arte resta alla pari.
9. *Le probabilità sono affidabili?* Soglie **conformali** per classe: perdono al più il 10% delle localizzazioni
   vere (con dati scambiabili); la copertura reale si verifica sul test.

Riferimento da battere (test HPA, DeepLoc 2.0 Tab. 2): ProtT5 → Accuracy 0.39, Jaccard 0.53, Micro F1 0.60, Macro F1 0.46.
Su CPU usare `ESM = 'facebook/esm2_t30_150M_UR50D'` e `N_CONFIGS = 4`.

In [7]:
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, f1_score, jaccard_score, matthews_corrcoef
from sklearn.model_selection import GroupKFold, cross_val_predict

DATA = Path('data')
ESM = 'facebook/esm2_t33_650M_UR50D'
DEVICE = ['cpu', 'cuda'][torch.cuda.is_available()]
CLASSES = ['Cytoplasm', 'Nucleus', 'Extracellular', 'Cell membrane', 'Mitochondrion',
           'Endoplasmic reticulum', 'Lysosome/Vacuole', 'Golgi apparatus', 'Peroxisome']
NEURO = ['Assone', 'Dendrite', 'Presinapsi', 'Postsinapsi']
ALL = CLASSES + NEURO
HPA_EVAL = ['Cytoplasm', 'Nucleus', 'Cell membrane', 'Mitochondrion', 'Endoplasmic reticulum', 'Golgi apparatus']
MAX_RES, TERM, WIN = 1022, 40, 15
MAX_VARIANTS = 4000
TOKENS = [2000, 12000][torch.cuda.is_available()]
SEED = 0
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

proteins = pd.read_parquet(DATA / 'proteine.parquet')
obs = pd.read_parquet(DATA / 'osservazioni.parquet')
cv = pd.read_parquet(DATA / 'clinvar_missense.parquet')
lab_var = pd.read_parquet(DATA / 'varianti_sperimentali.parquet')
AUX = list(proteins.columns[proteins.columns.str.startswith('aux_')])
CTX = list(proteins.columns[proteins.columns.str.startswith('ctx_')]) + ['brain_ntpm', 'neuron_ncpm', 'glia_ncpm']
print(proteins['source'].value_counts().to_string(), '\nTarget ausiliari:', AUX)

source
deeploc        28300
human_extra    10037
hpa_test        1700 
Target ausiliari: ['aux_peptide_segnale', 'aux_transit_peptide', 'aux_tm_1piu', 'aux_tm_2piu', 'aux_ancora_lipidica', 'aux_glicoproteina', 'aux_nls', 'aux_multimero', 'aux_fosforilazione', 'aux_ubiquitinazione', 'aux_palmitoilazione', 'aux_processamento', 'aux_aggregati', 'aux_patologia_cerebrale', 'aux_sinaptopatia']


## 1 · Sequenze da codificare: proteine + mutanti
Le varianti si tengono solo se cadono nella parte di sequenza che ESM vede (oltre 1022 aa si tengono i due terminali).
`center` è la posizione della variante nella sequenza troncata: serve per la vista locale.

In [8]:
seq_of = proteins.set_index('pid')['sequence']
L_wt = cv['pid'].map(seq_of.str.len())
cv = cv[(L_wt <= MAX_RES) | (cv['pos'] <= MAX_RES // 2) | (cv['pos'] > L_wt - MAX_RES // 2)].copy()
cv['priority'] = cv['clin'].map({'P/LP': 0, 'B/LB': 0, 'VUS': 1}) + rng.random(len(cv))
cv = cv.sort_values('priority').head(MAX_VARIANTS).reset_index(drop=True)
L_wt = cv['pid'].map(seq_of.str.len())
cv['pos_t'] = np.where((L_wt <= MAX_RES) | (cv['pos'] <= MAX_RES // 2), cv['pos'], cv['pos'] - L_wt + MAX_RES)
wt = cv['pid'].map(seq_of)
cv['mut_sequence'] = [s[:p - 1] + m + s[p:] for s, p, m in zip(wt, cv['pos'], cv['mt'])]

seqs = pd.concat([proteins['sequence'], cv['mut_sequence']], ignore_index=True)
N_PROT = len(proteins)
cv['row'] = np.arange(N_PROT, len(seqs))
center = np.full(len(seqs), -10 ** 6)
center[cv['row']] = cv['pos_t']
need_res = np.zeros(len(seqs), bool)
need_res[cv['pid'].unique()] = True                       # proteine WT di cui servono embedding per-residuo e logit
trunc = seqs.where(seqs.str.len() <= MAX_RES, seqs.str[:MAX_RES // 2] + seqs.str[-(MAX_RES // 2):])
print(f'Sequenze: {len(seqs):,} ({N_PROT:,} proteine + {len(cv):,} mutanti)')

Sequenze: 44,037 (40,037 proteine + 4,000 mutanti)


## 2 · Embedding ESM-2 (ripartibile)
Per ogni sequenza: media ultimo layer, media layer a 2/3, max, media primi 40 e ultimi 40 residui (`E`), più la media
locale ±15 residui attorno alla variante (`LOC`, solo mutanti). Per le proteine WT con varianti si salvano anche
l'embedding per-residuo e i log-probabilità di ESM (per vista locale WT e LLR).
Se cambi l'elenco delle sequenze cancella `data/emb_*` e `data/res_*`.

In [ ]:
from transformers import AutoTokenizer, EsmForMaskedLM

tok = AutoTokenizer.from_pretrained(ESM)
esm = EsmForMaskedLM.from_pretrained(ESM, torch_dtype=[torch.float32, torch.float16][DEVICE == 'cuda']).to(DEVICE).eval()
D, MID, tag = esm.config.hidden_size, round(esm.config.num_hidden_layers * 2 / 3), ESM.split('/')[-1]
AA = 'ACDEFGHIKLMNPQRSTVWY'
AA_IDS = tok.convert_tokens_to_ids(list(AA))
RES = DATA / f'res_{tag}'
RES.mkdir(exist_ok=True)

paths = {k: DATA / f'emb_{tag}_{k}.npy' for k in ['E', 'LOC', 'DONE']}

def _mm_shape(path):
    """Legge solo l'header .npy (file binario, nessun memmap aperto)."""
    with open(path, 'rb') as _f:
        _hdr = np.lib.format.read_magic(_f)
        _shape, _, _ = np.lib.format._read_array_header(_f, _hdr)
    return _shape

def _open_mm(path, dtype, shape):
    """Apre in r+ se la shape coincide; altrimenti cancella il file e ricrea con w+.
    Cancellare prima di scrivere evita l'OSError 22 di Windows sui file mmappati."""
    if path.exists() and _mm_shape(path) == shape:
        return np.lib.format.open_memmap(path, mode='r+', dtype=dtype, shape=shape)
    path.unlink(missing_ok=True)   # rimuove il vecchio file bloccato prima di ricrearlo
    return np.lib.format.open_memmap(path, mode='w+', dtype=dtype, shape=shape)

# rilascia eventuali memmap aperti in celle precedenti prima di riaprire
for _v in ['E', 'LOC', 'DONE']:
    if _v in dir(): del globals()[_v]
import gc; gc.collect()

E    = _open_mm(paths['E'],    np.float16, (len(seqs), 5 * D))
LOC  = _open_mm(paths['LOC'],  np.float16, (len(seqs), D))
DONE = _open_mm(paths['DONE'], bool,       (len(seqs),))

todo = trunc.str.len()[~DONE[:]].sort_values()
print(f'Da calcolare: {len(todo):,}')
for _, band in todo.groupby(todo // 64):
    batch_size = max(1, TOKENS // int(band.max() + 2))
    for start in range(0, len(band), batch_size):
        idx = band.index[start:start + batch_size].to_numpy()
        enc = tok(trunc[idx].tolist(), return_tensors='pt', padding=True).to(DEVICE)
        with torch.inference_mode():
            out = esm(**enc, output_hidden_states=True)
        L = enc['attention_mask'].sum(1, keepdim=True) - 2
        pos = torch.arange(enc['input_ids'].shape[1], device=DEVICE)[None]
        res = ((pos >= 1) & (pos <= L)).unsqueeze(-1)                      # residui veri (no BOS/EOS/padding)
        nterm, cterm = res & (pos <= TERM).unsqueeze(-1), res & (pos > L - TERM).unsqueeze(-1)
        local = res & ((pos - torch.tensor(center[idx], device=DEVICE)[:, None]).abs() <= WIN).unsqueeze(-1)
        last, mid = out.hidden_states[-1].float(), out.hidden_states[MID].float()
        E[idx] = torch.cat([(last * res).sum(1) / res.sum(1), (mid * res).sum(1) / res.sum(1),
                            last.masked_fill(~res, -1e4).max(1).values,
                            (last * nterm).sum(1) / nterm.sum(1), (last * cterm).sum(1) / cterm.sum(1)], 1).cpu().numpy().astype(np.float16)
        LOC[idx] = ((last * local).sum(1) / local.sum(1).clamp(min=1)).cpu().numpy().astype(np.float16)
        logp = out.logits.float().log_softmax(-1)[..., AA_IDS]
        for j in np.flatnonzero(need_res[idx]):
            n = int(L[j])
            np.savez(RES / f'{idx[j]}.npz', h=last[j, 1:n + 1].cpu().numpy().astype(np.float16), lp=logp[j, 1:n + 1].cpu().numpy())
        DONE[idx] = True
    for mm in (E, LOC, DONE):
        mm.flush()
print('Embedding completi:', DONE[:].all())

Some weights of the model checkpoint at facebook/esm2_t33_650M_UR50D were not used when initializing EsmForMaskedLM: ['esm.embeddings.position_embeddings.weight']
- This IS expected if you are initializing EsmForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Da calcolare: 44,037


## 3 · Feature biologiche dalla sequenza (valgono anche per i mutanti)

In [ ]:
KD = dict(zip(AA, [1.8, 2.5, -3.5, -3.5, 2.8, -0.4, -3.2, 4.5, -3.9, 3.8, 1.9, -3.5, -1.6, -3.5, -4.5, -0.8, -0.7, 4.2, -0.9, -1.3]))
n_len, nt, ct = seqs.str.len(), seqs.str[:30], seqs.str[-30:]
hand = pd.DataFrame({f'aa_{a}': seqs.str.count(a) / n_len for a in AA})
hand['log_len'] = np.log(n_len)
hand['nt_kd'] = sum(nt.str.count(a) * KD[a] for a in AA) / nt.str.len()
hand['nt_charge'] = (nt.str.count('[KR]') - nt.str.count('[DE]')) / 30      # presequenza mitocondriale: positiva
hand['nt_hydro_core'] = nt.str.contains(r'[AILMFVWC]{7,}').astype(int)     # nucleo idrofobico del peptide segnale
hand['tm_like'] = seqs.str.count(r'[^DEKRHNQPS]{17,}')
hand['nls'] = seqs.str.count(r'K[KR].[KR]') + seqs.str.count(r'[KR]{2}.{10,12}[KR]{3}')
hand['kdel'] = seqs.str.contains(r'[KRHQSA][DENQ]EL$').astype(int)
hand['kkxx'] = seqs.str.contains(r'K.K..$|KK..$').astype(int)
hand['pts1'] = seqs.str.contains(r'[SAC][KRH][LM]$').astype(int)
hand['caax'] = seqs.str.contains(r'C[AILMV][AILMV].$').astype(int)
hand['myr'] = seqs.str.contains(r'^MG[^EDRKHPFYW]..[STAGCN][^P]').astype(int)
hand['ct_kd'] = sum(ct.str.count(a) * KD[a] for a in AA) / ct.str.len()
BLOCKS = {'mean_last': E[:, :D], 'mean_mid': E[:, D:2 * D], 'max': E[:, 2 * D:3 * D],
          'nterm': E[:, 3 * D:4 * D], 'cterm': E[:, 4 * D:], 'hand': hand.to_numpy(np.float16)}

## 4 · Target: osservazioni per fonte, target ausiliari, partizioni
`OBS[proteina, fonte, classe]` vale 1, 0 o NaN (non osservato). La fonte `deeploc` è l'ancora.
Le osservazioni neuronali contano solo per proteine espresse nei neuroni.
Le proteine umane aggiunte ereditano la partizione della proteina DeepLoc più simile (coseno sull'embedding medio).

In [ ]:
SOURCES = ['deeploc'] + sorted(set(obs['fonte']) - {'deeploc'})
S, C, A, nC = len(SOURCES), len(ALL), len(AUX), len(CLASSES)
OBS = np.full((N_PROT, S, C), np.nan, np.float32)
OBS[obs['pid'].to_numpy(), obs['fonte'].map({s: i for i, s in enumerate(SOURCES)}).to_numpy(),
    obs['classe'].map({c: i for i, c in enumerate(ALL)}).to_numpy()] = obs['obs'].to_numpy()
OBS[proteins['is_neuronal'].to_numpy() == 0, :, nC:] = np.nan
AUXY = proteins[AUX].to_numpy(np.float32)

src = proteins['source'].to_numpy()
part = proteins['partition'].to_numpy().astype(int)
is_brain, is_neur = proteins['is_brain'].to_numpy().astype(bool), proteins['is_neuronal'].to_numpy().astype(bool)
test_rows = np.flatnonzero(src == 'hpa_test')
Y_test = proteins.loc[test_rows].reindex(columns=CLASSES).fillna(0).to_numpy(np.float32)
GOLD = OBS[:, 0, :nC]                                                    # etichette DeepLoc (NaN fuori da DeepLoc)

unit = BLOCKS['mean_last'][:N_PROT].astype(np.float32)
unit /= np.linalg.norm(unit, axis=1, keepdims=True)
dl_rows, extra_rows = np.flatnonzero(src == 'deeploc'), np.flatnonzero(src == 'human_extra')
nearest = np.concatenate([(unit[c] @ unit[dl_rows].T).argmax(1) for c in np.array_split(extra_rows, max(1, len(extra_rows) // 2000))])
part[extra_rows] = part[dl_rows[nearest]]

# controlli interni: nessuna sequenza del test nel training, vicinanza test-training, copertura delle fonti
assert not set(proteins.loc[test_rows, 'sequence']) & set(proteins.loc[np.r_[dl_rows, extra_rows], 'sequence'])
sim_test = np.concatenate([(unit[c] @ unit[dl_rows].T).max(1) for c in np.array_split(test_rows, max(1, len(test_rows) // 1000))])
print(f'Coseno max test→training: mediana {np.median(sim_test):.3f}, quota > 0.95: {(sim_test > 0.95).mean():.1%}')
print(pd.DataFrame({'positivi': np.nansum(OBS, axis=(0, 2)), 'osservati': (~np.isnan(OBS)).sum((0, 2))}, index=SOURCES).astype(int))

## 5 · Grid search con CV a 5 fold sulle partizioni di omologia
Pool di training (proteine × fonti):
- `oro` → solo DeepLoc, solo etichette DeepLoc: riproduce il protocollo dello stato dell'arte;
- `oro+fonti` → stesse proteine, tutte le fonti (canale di rumore attivo): misura il guadagno delle fonti sul test HPA;
- `esteso` → anche le proteine umane aggiunte, con canale di rumore (`rumore=1`) o fusione ingenua (`rumore=0`):
  è l'ablazione che dice se lo strato di rumore serve.
Ogni fold: test = partizione k, validazione = k+1 (early stopping sulla loss DeepLoc), training = le altre.
Canale: P(la fonte dice 1) = sens·p + (1−spec)·(1−p), con p = probabilità latente; `rumore=0` forza sens = spec = 1.

In [ ]:
BLOCK_SETS = [('mean_last', 'mean_mid', 'nterm', 'cterm', 'hand'), ('mean_last', 'mean_mid', 'max', 'nterm', 'cterm', 'hand')]
GRID = dict(blocks=BLOCK_SETS, hidden=[512, 1024], dropout=[0.3, 0.5], lr=[5e-4, 1e-3], aux_weight=[0.0, 0.3])
POOLS = pd.DataFrame([('oro', 0), ('oro+fonti', 1), ('esteso', 0), ('esteso', 1)], columns=['pool', 'rumore'])
N_CONFIGS, EPOCHS, BATCH, WD = 8, 30, 128, 1e-2

configs = pd.DataFrame(list(product(*GRID.values())), columns=list(GRID)).sample(N_CONFIGS, random_state=SEED)
runs = configs.merge(POOLS, how='cross').reset_index(drop=True)
var_rows = np.r_[cv['pid'].unique(), cv['row']]
OBS_t = torch.tensor(np.nan_to_num(OBS), device=DEVICE)
AUX_t = torch.tensor(np.nan_to_num(AUXY), device=DEVICE)
AUXM_t = torch.tensor(~np.isnan(AUXY), device=DEVICE).float()

PRED, CHANNEL = [], []
for r, run in runs.iterrows():
    X = torch.tensor(np.hstack([BLOCKS[b] for b in run['blocks']]))
    rows_pool = (src == 'deeploc') | ((src == 'human_extra') & (run['pool'] == 'esteso'))
    src_on = torch.tensor([1.0] + [float(run['pool'] != 'oro')] * (S - 1), device=DEVICE)   # fonti usate dal pool
    MASK = torch.tensor(~np.isnan(OBS), device=DEVICE).float() * src_on[None, :, None]
    oof = np.full((N_PROT, C + A), np.nan, np.float32)
    hpa_pred, var_pred, chan = [], [], []
    for k in range(5):
        tr = np.flatnonzero(rows_pool & (part != k) & (part != (k + 1) % 5))
        va, te = np.flatnonzero(rows_pool & (part == (k + 1) % 5)), np.flatnonzero(rows_pool & (part == k))
        mu, sd = X[tr].float().mean(0), X[tr].float().std(0) + 1e-4
        pos_n = (OBS_t[tr] * MASK[tr]).sum((0, 1))
        pos_weight = ((MASK[tr].sum((0, 1)) - pos_n) / pos_n.clamp(min=1)).sqrt().clamp(1, 10)
        torch.manual_seed(SEED + k)
        net = nn.ModuleDict({'trunk': nn.Sequential(nn.Dropout(0.1), nn.Linear(X.shape[1], run['hidden']), nn.GELU(),
                                                    nn.Dropout(run['dropout'])),
                             'loc': nn.Linear(run['hidden'], C), 'aux': nn.Linear(run['hidden'], A)}).to(DEVICE)
        # canale per fonte, inizializzato dai dati: sulle proteine di training osservate sia dalla fonte sia da
        # DeepLoc, sensibilità e specificità empiriche con un prior Beta (10 pseudo-conteggi a 0.85 / 0.97).
        # Le classi neuronali (senza DeepLoc) partono dal prior. DeepLoc (fonte 0) resta perfetta.
        g, gm = np.nan_to_num(GOLD[tr])[:, None, :], ~np.isnan(GOLD[tr])[:, None, :]
        o_tr, om = np.nan_to_num(OBS[tr, :, :nC]), ~np.isnan(OBS[tr, :, :nC]) & gm
        sens0 = np.full((S, C), 0.85)
        spec0 = np.full((S, C), 0.97)
        sens0[:, :nC] = ((o_tr * g * om).sum(0) + 8.5) / ((g * om).sum(0) + 10)
        spec0[:, :nC] = (((1 - o_tr) * (1 - g) * om).sum(0) + 9.7) / (((1 - g) * om).sum(0) + 10)
        sens_raw = nn.Parameter(torch.tensor(np.log(sens0 / (1 - sens0)), dtype=torch.float32, device=DEVICE))
        spec_raw = nn.Parameter(torch.tensor(np.log(spec0 / (1 - spec0)), dtype=torch.float32, device=DEVICE))
        free = torch.full((S, 1), float(run['rumore']), device=DEVICE)
        free[0] = 0
        # il canale ha pochi parametri e deve muoversi più in fretta della rete: learning rate 10x, niente weight decay
        opt = torch.optim.AdamW([{'params': net.parameters(), 'lr': run['lr'], 'weight_decay': WD},
                                 {'params': [sens_raw, spec_raw], 'lr': 10 * run['lr'], 'weight_decay': 0.0}])
        steps = EPOCHS * int(np.ceil(len(tr) / BATCH))
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[run['lr'], 10 * run['lr']], total_steps=steps, pct_start=0.1)
        Xn = {n: ((X[rr].float() - mu) / sd).to(DEVICE) for n, rr in [('va', va), ('te', te), ('hpa', test_rows), ('var', var_rows)]}
        hist = {'val': [], 'te': [], 'hpa': [], 'var': []}
        for ep in range(EPOCHS):
            net.train()
            for b in np.array_split(rng.permutation(tr), int(np.ceil(len(tr) / BATCH))):
                h = net['trunk'](((X[b].float() - mu) / sd).to(DEVICE))
                p = torch.sigmoid(net['loc'](h))[:, None, :].clamp(1e-5, 1 - 1e-5)
                sens = 1 - free * (1 - torch.sigmoid(sens_raw))
                spec = 1 - free * (1 - torch.sigmoid(spec_raw))
                q = (sens * p + (1 - spec) * (1 - p)).clamp(1e-5, 1 - 1e-5)          # P(la fonte dice 1)
                o, mk = OBS_t[b], MASK[b]
                loss_loc = -(mk * (pos_weight * o * q.log() + (1 - o) * (1 - q).log())).sum() / mk.sum().clamp(min=1)
                loss_aux = (AUXM_t[b] * F.binary_cross_entropy_with_logits(net['aux'](h), AUX_t[b], reduction='none')).sum() / AUXM_t[b].sum().clamp(min=1)
                loss = loss_loc + run['aux_weight'] * loss_aux
                opt.zero_grad()
                loss.backward()
                opt.step()
                sched.step()
            net.eval()
            with torch.no_grad():
                m_va = MASK[va, 0, :nC]
                bce_va = F.binary_cross_entropy_with_logits(net['loc'](net['trunk'](Xn['va']))[:, :nC], OBS_t[va, 0, :nC], reduction='none')
                hist['val'].append(((m_va * bce_va).sum() / m_va.sum().clamp(min=1)).item())
                for n in ['te', 'hpa', 'var']:
                    h_n = net['trunk'](Xn[n])
                    hist[n].append(torch.cat([net['loc'](h_n), net['aux'](h_n)], 1).cpu().numpy())
        best = int(np.argmin(hist['val']))
        oof[te] = hist['te'][best]
        hpa_pred.append(hist['hpa'][best])
        var_pred.append(hist['var'][best])
        chan.append(np.stack([torch.sigmoid(sens_raw).detach().cpu().numpy(), torch.sigmoid(spec_raw).detach().cpu().numpy()]))
    gold_ok = (src == 'deeploc') & ~np.isnan(oof[:, 0])
    score = np.mean([average_precision_score(GOLD[gold_ok, c], oof[gold_ok, c]) for c in range(nC)])
    PRED.append({'oof': oof, 'hpa': np.mean(hpa_pred, 0), 'var': np.stack(var_pred)})
    CHANNEL.append(np.mean(chan, 0))
    runs.loc[r, 'cv_macro_auprc'] = score
    print(f"[{r + 1}/{len(runs)}] {run['pool']:<10} rumore={run['rumore']} {'+'.join(run['blocks']):<42} h={run['hidden']} "
          f"drop={run['dropout']} lr={run['lr']} aux={run['aux_weight']} → macro-AUPRC (oro, CV) {score:.4f}")

## 6 · Cosa dice la grid search: ablazioni e affidabilità delle fonti
Il punteggio è sempre la macro-AUPRC out-of-fold sulle etichette DeepLoc, quindi i pool sono confrontabili.
La tabella di affidabilità è essa stessa un risultato: quanto è completa e quanto è precisa ogni banca dati,
classe per classe, stimato dai dati.

In [ ]:
runs['blocks_str'] = runs['blocks'].map('+'.join)
print(runs.groupby(['pool', 'rumore'])['cv_macro_auprc'].agg(['mean', 'max']).round(4), '\n')
for col in ['blocks_str', 'hidden', 'dropout', 'lr', 'aux_weight']:
    print(runs.groupby(col)['cv_macro_auprc'].agg(['mean', 'max']).round(4), '\n')

chan = np.mean([CHANNEL[i] for i in runs.index[runs['rumore'] == 1]], 0)
seen = ~np.isnan(OBS).all(0)
reliab = pd.DataFrame({'sensibilità media': [chan[0, s][seen[s]].mean() for s in range(S)],
                       'specificità media': [chan[1, s][seen[s]].mean() for s in range(S)]}, index=SOURCES)
reliab.loc['deeploc'] = 1.0
print(reliab.round(3))
print('\nSensibilità per classe:')
print(pd.DataFrame(chan[0], index=SOURCES, columns=ALL).where(seen).round(2).T)

## 7 · Ensemble, k-NN, soglie MCC/Jaccard e soglie conformali (tutto scelto sull'OOF)
Per ogni pool: ensemble dei run entro `TOL` dal migliore, blending con k-NN sugli embedding (α scelto sull'OOF),
soglie MCC per classe, versione riscalata che massimizza il Jaccard, soglie conformali al 90% di copertura.

In [ ]:
TOP_K, TOL, KNN, EPS = 3, 0.01, 10, 0.10
ALPHAS, SCALES, grid_t = np.linspace(0, 0.8, 17), np.linspace(0.8, 3.2, 49), np.linspace(0.05, 0.95, 91)
FINAL = {}
for (pool_name, rum), rr in runs.groupby(['pool', 'rumore']):
    top = rr[rr['cv_macro_auprc'] >= rr['cv_macro_auprc'].max() - TOL].sort_values('cv_macro_auprc', ascending=False).index[:TOP_K]
    P_oof = np.mean([1 / (1 + np.exp(-PRED[i]['oof'])) for i in top], 0)
    P_hpa = np.mean([1 / (1 + np.exp(-PRED[i]['hpa'])) for i in top], 0)
    ok = np.flatnonzero((src == 'deeploc') & ~np.isnan(P_oof[:, 0]))
    knn_oof = np.zeros((N_PROT, nC))
    for k in range(5):
        ref, qry = ok[part[ok] != k], ok[part[ok] == k]
        sim = unit[qry] @ unit[ref].T
        nb = np.argpartition(-sim, KNN, axis=1)[:, :KNN]
        w = np.take_along_axis(sim, nb, 1).clip(0) ** 8                 # i vicini molto simili pesano molto di più
        knn_oof[qry] = (w[..., None] * GOLD[ref][nb]).sum(1) / (w.sum(1, keepdims=True) + 1e-9)
    sim = unit[test_rows] @ unit[ok].T
    nb = np.argpartition(-sim, KNN, axis=1)[:, :KNN]
    w = np.take_along_axis(sim, nb, 1).clip(0) ** 8
    knn_hpa = (w[..., None] * GOLD[ok][nb]).sum(1) / (w.sum(1, keepdims=True) + 1e-9)
    a_score = [np.mean([average_precision_score(GOLD[ok, c], (1 - a) * P_oof[ok, c] + a * knn_oof[ok, c]) for c in range(nC)]) for a in ALPHAS]
    alpha = ALPHAS[int(np.argmax(a_score))]
    P_oof[:, :nC] = (1 - alpha) * P_oof[:, :nC] + alpha * knn_oof
    P_hpa[:, :nC] = (1 - alpha) * P_hpa[:, :nC] + alpha * knn_hpa
    Pg, Yg = P_oof[ok, :nC], GOLD[ok].astype(int)
    thr_mcc = np.array([grid_t[np.argmax([matthews_corrcoef(Yg[:, c], Pg[:, c] >= t) for t in grid_t])] for c in range(nC)])
    jac = [jaccard_score(Yg, np.maximum(Pg >= (thr_mcc * s).clip(0, .99), Pg == Pg.max(1, keepdims=True)).astype(int), average='samples') for s in SCALES]
    thr_jac = (thr_mcc * SCALES[int(np.argmax(jac))]).clip(0, .99)
    # conformale per classe: soglia = quantile ε delle probabilità OOF dei veri positivi
    thr_conf = np.array([np.quantile(Pg[Yg[:, c] == 1, c], EPS) for c in range(nC)])
    FINAL[(pool_name, rum)] = dict(top=top, alpha=alpha, P_oof=P_oof, P_hpa=P_hpa, thr_mcc=thr_mcc, thr_jac=thr_jac, thr_conf=thr_conf)
    print(f'{pool_name:<10} rumore={rum}: run {list(top)}, α k-NN {alpha:.2f}, scala soglie {SCALES[int(np.argmax(jac))]:.2f}')

## 8 · Confronto con lo stato dell'arte sul test HPA indipendente
`oro` = stesso protocollo di DeepLoc 2.0; `oro+fonti` = stesse proteine con le fonti aggiuntive.

In [ ]:
ev = [CLASSES.index(c) for c in HPA_EVAL]
y_true = Y_test[:, ev].astype(int)
sota = pd.DataFrame(index=['Labels predette', 'Accuracy', 'Jaccard', 'Micro F1', 'Macro F1'] + [f'MCC {c}' for c in HPA_EVAL] + ['Copertura conformale'])
for key in [('oro', 0), ('oro+fonti', 1)]:
    res = FINAL[key]
    Ph = res['P_hpa'][:, :nC]
    cover = ((Ph >= res['thr_conf']) & (Y_test == 1))[:, ev].sum() / (Y_test[:, ev] == 1).sum()
    for name, t in [('MCC', res['thr_mcc']), ('Jaccard', res['thr_jac'])]:
        y_pred = np.maximum(Ph >= t, Ph == Ph.max(1, keepdims=True)).astype(int)[:, ev]
        sota[f'{key[0]} · soglie {name}'] = [y_pred.sum(1).mean(), (y_true == y_pred).all(1).mean(),
                                             jaccard_score(y_true, y_pred, average='samples'), f1_score(y_true, y_pred, average='micro'),
                                             f1_score(y_true, y_pred, average='macro')] + \
                                            [matthews_corrcoef(y_true[:, j], y_pred[:, j]) for j in range(len(ev))] + [cover]
sota['DeepLoc 2.0 ESM1b'] = [1.15, 0.34, 0.48, 0.57, 0.44, 0.29, 0.41, 0.34, 0.60, 0.20, 0.17, np.nan]
sota['DeepLoc 2.0 ProtT5'] = [1.21, 0.39, 0.53, 0.60, 0.46, 0.36, 0.44, 0.36, 0.56, 0.17, 0.31, np.nan]
print(f'Test HPA: {len(test_rows):,} proteine, etichette reali per proteina {y_true.sum(1).mean():.2f}')
print(sota.round(3).to_string())
sota.round(3).to_excel(DATA / 'confronto_stato_arte.xlsx')

## 9 · Cervello e compartimenti neuronali (pool `esteso`)
Si usa la variante (rumore sì/no) migliore in CV. Riferimento per la valutazione = fonti forti (DeepLoc, UniProt
forte, HPA alta) per gli organelli, qualsiasi fonte neuronale per assone/dendrite/sinapsi. Tutto out-of-fold.
Organelli valutati sulle proteine cerebrali, compartimenti neuronali sulle proteine espresse nei neuroni.

In [ ]:
best_rum = int(runs[runs['pool'] == 'esteso'].groupby('rumore')['cv_macro_auprc'].max().idxmax())
res_b = FINAL[('esteso', best_rum)]
Pb = res_b['P_oof']
ref_src = np.flatnonzero(np.isin(SOURCES, ['deeploc', 'uniprot_forte', 'hpa_alta']))
REF = np.where(np.isnan(OBS[:, ref_src]), -1, OBS[:, ref_src]).max(axis=1)          # -1 = nessuna fonte forte
REF[:, nC:] = np.where(np.isnan(OBS[:, :, nC:]), -1, OBS[:, :, nC:]).max(axis=1)
in_pool = ~np.isnan(Pb[:, 0])
masks = [in_pool & (REF[:, c] >= 0) & [is_brain, is_neur][int(c >= nC)] for c in range(C)]
with np.errstate(all='ignore'):
    report = pd.DataFrame({'proteine': [mk.sum() for mk in masks],
                       'positivi': [(REF[mk, c] == 1).sum() for c, mk in enumerate(masks)],
                       # AUPRC esplicita (precisione media sui positivi): NaN invece di errore se una classe non ha positivi
                       'AUPRC': [(np.cumsum(y_s) / np.arange(1, len(y_s) + 1) * y_s).sum() / y_s.sum()
                                 for c, mk in enumerate(masks) for y_s in [(REF[mk, c] == 1)[np.argsort(-Pb[mk, c])]]]}, index=ALL)
report['prevalenza (= AUPRC casuale)'] = report['positivi'] / report['proteine']
print(f'Pool esteso, rumore={best_rum}')
print(report.round(3).to_string())

bmask = in_pool & is_brain
table = proteins.loc[bmask, ['pid', 'uniprot_id', 'gene', 'source', 'is_neuronal']].copy()
table['predette'] = ['; '.join(np.array(CLASSES)[np.maximum(p >= res_b['thr_jac'], p == p.max())]) for p in Pb[bmask, :nC]]
table['insieme_conformale_90'] = ['; '.join(np.array(CLASSES)[p >= res_b['thr_conf']]) for p in Pb[bmask, :nC]]
table[[f'P {c}' for c in ALL]] = Pb[bmask, :C].round(4)
table[[f'% {c}' for c in CLASSES]] = (100 * Pb[bmask, :nC] / Pb[bmask, :nC].sum(1, keepdims=True)).round(1)
table.to_excel(DATA / 'probabilita_proteine_cervello.xlsx', index=False)

## 10 · NeuroLoc-Δ: meccanismi di mislocalizzazione delle varianti
WT e mutante sono predetti dal modello del fold in cui la proteina WT era nel test (nessuna memorizzazione).
Segnali per variante:
- **Δ organelli** (JSD, massima variazione) e **Δ compartimenti neuronali**;
- **Δ meccanismi**: variazione delle teste ausiliarie (peptide segnale, TM, ancora lipidica, NLS, glicosilazione...);
- **spostamento locale**: 1 − coseno tra embedding ±15 residui di WT e mutante;
- **LLR di ESM** (log P mutante − log P WT, contesto WT): proxy di destabilizzazione;
- **ΔΔG di inserzione in membrana** (scala biologica di Hessa) se la variante cade in una finestra TM-compatibile;
- contesto genico (Excel FFFFF e ClinVar).
Due punteggi: **non supervisionato** (media dei ranghi dei segnali di meccanismo) e **supervisionato**
(gradient boosting su ClinVar P/LP vs B/LB, CV raggruppata per gene). Entrambi valutati anche sulle varianti con
mislocalizzazione misurata, se presenti.

In [ ]:
var_P = np.mean([1 / (1 + np.exp(-PRED[i]['var'])) for i in res_b['top']], 0)        # [fold, riga, uscita]
vi = pd.Series(np.arange(len(var_rows)), index=var_rows)
fold = part[cv['pid']]
p_wt, p_mt = var_P[fold, vi[cv['pid']].to_numpy()], var_P[fold, vi[cv['row']].to_numpy()]
q_wt, q_mt = p_wt[:, :nC] / p_wt[:, :nC].sum(1, keepdims=True), p_mt[:, :nC] / p_mt[:, :nC].sum(1, keepdims=True)
q_m = (q_wt + q_mt) / 2
cv['JSD'] = 0.5 * (q_wt * np.log2(q_wt / q_m)).sum(1) + 0.5 * (q_mt * np.log2(q_mt / q_m)).sum(1)
cv['max_dP_organelli'] = np.abs(p_mt[:, :nC] - p_wt[:, :nC]).max(1)
cv['max_dP_neuro'] = np.abs(p_mt[:, nC:C] - p_wt[:, nC:C]).max(1)
cv[[f'd_{a}' for a in AUX]] = p_mt[:, C:] - p_wt[:, C:]
cv['max_d_meccanismo'] = np.abs(p_mt[:, C:] - p_wt[:, C:]).max(1)
cv['meccanismo_principale'] = np.array(AUX)[np.abs(p_mt[:, C:] - p_wt[:, C:]).argmax(1)]

# vista locale WT (embedding per-residuo salvati) contro vista locale del mutante; LLR dai log-prob WT
res_wt = {pid: np.load(RES / f'{pid}.npz') for pid in cv['pid'].unique()}
loc_wt = np.stack([res_wt[pid]['h'][max(pt - 1 - WIN, 0):pt + WIN].astype(np.float32).mean(0) for pid, pt in zip(cv['pid'], cv['pos_t'])])
loc_mt = LOC[cv['row']].astype(np.float32)
cv['shift_locale'] = 1 - (loc_wt * loc_mt).sum(1) / (np.linalg.norm(loc_wt, axis=1) * np.linalg.norm(loc_mt, axis=1))
aa_i = {a: i for i, a in enumerate(AA)}
cv['LLR'] = [res_wt[pid]['lp'][pt - 1, aa_i[m_]] - res_wt[pid]['lp'][pt - 1, aa_i[w_]]
             for pid, pt, w_, m_ in zip(cv['pid'], cv['pos_t'], cv['wt'], cv['mt'])]

# biofisica di membrana: ΔG apparente di inserzione (Hessa et al., Nature 2005) su una finestra di 19 residui
HESSA = dict(zip('ILFVMCAWTYGSNHPQREKD', [-0.60, -0.55, -0.32, -0.31, -0.10, -0.13, 0.11, 0.30, 0.52, 0.68, 0.74, 0.84,
                                          2.05, 2.06, 2.23, 2.36, 2.58, 2.68, 2.71, 3.49]))
win19 = pd.Series([s[max(p - 10, 0):p + 9] for s, p in zip(wt, cv['pos'])], index=cv.index)
dG_wt = sum(win19.str.count(a) * HESSA[a] for a in HESSA)
in_tm = (dG_wt < 2.0) & (win19.str.len() == 19)
cv['ddG_inserzione_TM'] = (cv['mt'].map(HESSA) - cv['wt'].map(HESSA)) * in_tm
cv['perde_inserzione'] = (in_tm & (dG_wt < 0) & (dG_wt + cv['ddG_inserzione_TM'] > 0)).astype(int)
cv['in_Nterm30'] = (cv['pos'] <= 30).astype(int)
cv['in_Cterm10'] = (cv['pos'] > wt.str.len() - 10).astype(int)
hand_d = hand.iloc[cv['row']].to_numpy() - hand.iloc[cv['pid']].to_numpy()
cv['motivi_alterati'] = np.abs(hand_d[:, [list(hand.columns).index(c) for c in ['kdel', 'kkxx', 'pts1', 'caax', 'myr', 'nls']]]).sum(1)
cv['d_nt_charge'] = hand_d[:, list(hand.columns).index('nt_charge')]
cv = cv.join(proteins.set_index('pid')[CTX], on='pid')

# punteggio non supervisionato: media dei ranghi dei segnali di meccanismo (più alto = più sospetto)
MECH = ['JSD', 'max_dP_neuro', 'max_d_meccanismo', 'shift_locale', 'ddG_inserzione_TM', 'motivi_alterati']
cv['score_misloc'] = np.mean([rankdata(cv[c].abs()) / len(cv) for c in MECH] + [rankdata(-cv['LLR']) / len(cv)], 0)

# punteggio supervisionato su ClinVar, CV raggruppata per gene (nessun gene in training e test insieme)
FEAT = MECH + ['max_dP_organelli', 'LLR', 'perde_inserzione', 'in_Nterm30', 'in_Cterm10', 'd_nt_charge'] + \
       [f'd_{a}' for a in AUX] + CTX + ['ctx_frazione_patogeniche_gene']
pb = cv[cv['clin'].isin(['P/LP', 'B/LB'])]
FEAT = list(pb[FEAT].columns[pb[FEAT].nunique() > 1])          # via le colonne vuote o costanti (es. Excel assente)
y_pb = (pb['clin'] == 'P/LP').astype(int).to_numpy()
hgb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0, random_state=SEED)
cv['score_supervisionato'] = np.nan
cv.loc[pb.index, 'score_supervisionato'] = cross_val_predict(hgb, pb[FEAT].astype(float), y_pb, groups=pb['gene'],
                                                             cv=GroupKFold(5), method='predict_proba')[:, 1]
# le VUS ricevono il punteggio del modello allenato su tutte le P/LP e B/LB
full = hgb.fit(pb[FEAT].astype(float), y_pb).predict_proba(cv[FEAT].astype(float))[:, 1]
cv['score_supervisionato'] = np.where(cv['clin'] == 'VUS', full, cv['score_supervisionato'])

# AUROC con la formula dei ranghi: dà NaN (senza errori) se mancano dati, es. nessuna variante sperimentale
lv = cv[['gene', 'variante']].reset_index().merge(lab_var.drop_duplicates(['gene', 'variante']), on=['gene', 'variante'])
EVAL = {f'ClinVar P/LP vs B/LB (n={len(pb)})': (pb.index.to_numpy(), y_pb),
        f'Mislocalizzazione misurata (n={len(lv)})': (lv['index'].to_numpy(), lv['mislocalizzata'].astype(int).to_numpy())}
SCORES = {'JSD': cv['JSD'], 'shift_locale': cv['shift_locale'], 'max_d_meccanismo': cv['max_d_meccanismo'],
          'LLR (segno −)': -cv['LLR'], '|ddG_inserzione_TM|': cv['ddG_inserzione_TM'].abs(),
          'score_misloc': cv['score_misloc'], 'score_supervisionato': cv['score_supervisionato']}
auc = pd.DataFrame(index=list(SCORES))
for name, (ix, y) in EVAL.items():
    with np.errstate(all='ignore'):
        auc[name] = [(rankdata(v[ix])[y == 1].sum() - (y == 1).sum() * ((y == 1).sum() + 1) / 2) / ((y == 1).sum() * (y == 0).sum())
                     for v in SCORES.values()]
print('AUROC:')
print(auc.round(3).to_string())
print('\nMeccanismo principale nelle P/LP più sospette (top 10% di score_misloc):')
print(cv[(cv['clin'] == 'P/LP') & (cv['score_misloc'] >= cv['score_misloc'].quantile(0.9))]['meccanismo_principale'].value_counts().head(8).to_string())

cv.drop(columns=['mut_sequence', 'priority']).sort_values('score_misloc', ascending=False).to_excel(DATA / 'varianti_mislocalizzazione.xlsx', index=False)